This iPythonNotebook can be used to experiment with the Post Training Calibration based method of quantization, where the user will be able to introduce quantization to any network. Further, the user can use the quantized model for evaluations on the pytorch framework itself. 

In [ ]:
!pip install netron

import torch
import torch.nn as nn
import edgeai_torchmodelopt
import copy
import netron
import torchvision
from tqdm import tqdm

In [ ]:
model = torchvision.models.resnet50(weights='DEFAULT')
example_input = torch.rand((1, 3, 224, 224))

y = model(example_input)
print("Output Shape is : {}".format(y.shape))

In [ ]:
model_export_name = "./orig_simple_network_ptc.onnx"
torch.onnx.export(model, example_input, model_export_name)
netron.start(model_export_name, 8080)

Here we will be wrapping our model in the PTCFxModule which will be responsible for the calibration of the models and conversion to the final quantized network. It also enables bias calibration of the layers having a bias value, we can set a bias calibration factor (generally 0.01 works well) to enable it. Further, num_batch_norm_update_epochs and num_observer_update_epochs are used to define the epochs for which batch norm params and the observer are updated respectively. Each epoch is updated when a call to model.train() is done. Calibration is suggested to be performed in the training mode to utilise full functionality. Here, we do calibration for 3 epochs, and it can be done for very small examples from the distribution (generally 100 is good enough).   

In [ ]:
model = edgeai_torchmodelopt.xmodelopt.quantization.v2.PTCFxModule(model, backend='qnnpack', bias_calibration_factor=0.01, num_batch_norm_update_epochs=1, num_observer_update_epochs=2)

Here is the Calibration Step for the network, where random data is used currently just for an example. **The data should be changed to your own dataset.**

In [ ]:
num_calib_images = 10
num_epochs = 3
for epoch in range(num_epochs):
    model.train()
    for i in tqdm(range(num_calib_images)):
        output = model(torch.rand(1,3,224,224))

We have the quantized and calibrated 8-bit network now.

In [ ]:
print(model)

In [ ]:
model_export_name = "./converted_simple_network_ptc.onnx"
model.export(example_input, model_export_name)
netron.start(model_export_name, 8080)


The netron might show the quantized fused operators as separate because the fake-quantized (Q-DQ) models are exported. 